# PushT End-to-End: DINOv2 World Model

Full pipeline: data collection → train a DINOv2-based world model → evaluate with MPC.

**Architecture**: frozen DINOv2 ViT-S/14 backbone → ConvNet spatial adapter (384→128, 16×16→4×4) → 2D positional embedding → 1-layer Transformer predictor → CEM planning.

**Training**: multi-step rollout MSE in DINOv2 embedding space.  
**Eval**: CEMSolver + WorldModelPolicy, cost = cosine distance to goal embedding.

In [ ]:
# Cell 1: Imports & device
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms as T
from transformers import Dinov2Model

import stable_worldmodel as swm
from stable_worldmodel.policy import WorldModelPolicy, PlanConfig
from stable_worldmodel.solver import CEMSolver

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 2: Data collection (skip if already collected)
DATA_PATH = 'data/pusht_demo.lance'

if not os.path.exists(DATA_PATH):
    print('Collecting data...')
    world = swm.World('swm/PushT-v1', num_envs=8, image_shape=(64, 64))
    world.set_policy(swm.policy.RandomPolicy(seed=44))
    world.collect(DATA_PATH, episodes=100, seed=0)
    world.close()
    print('Done.')
else:
    print(f'Dataset already exists at {DATA_PATH}')

In [ ]:
# Cell 3: Dataset loading
NUM_STEPS = 16  # trajectory length per training sample
BATCH_SIZE = 16

abs_path = os.path.abspath(DATA_PATH)
dataset = swm.data.load_dataset(
    abs_path,
    frameskip=1,
    num_steps=NUM_STEPS,
    keys_to_load=['pixels', 'action'],
)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

sample = dataset[0]
print(f'Dataset size:  {len(dataset)} samples')
print(f'Pixels shape:  {sample["pixels"].shape}')   # (T, 3, 64, 64)
print(f'Action shape:  {sample["action"].shape}')   # (T, 2)
print(f'Pixels dtype:  {sample["pixels"].dtype}')
print(f'Pixels range:  [{sample["pixels"].min()}, {sample["pixels"].max()}]')

In [ ]:
# Cell 4: Model definitions

# ─── DINOv2 encoder ────────────────────────────────────────────────────────────
class DinoEncoder(nn.Module):
    """Frozen DINOv2 ViT-S/14 backbone: image → spatial patch features (B, 384, 16, 16)."""

    PATCH_H = PATCH_W = 16  # number of patches per side at 224×224 with patch_size=14

    def __init__(self):
        super().__init__()
        self.dino = Dinov2Model.from_pretrained('facebook/dinov2-small')
        for p in self.dino.parameters():
            p.requires_grad = False
        self.register_buffer(
            'img_mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        )
        self.register_buffer(
            'img_std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        )

    def preprocess(self, x: torch.Tensor) -> torch.Tensor:
        """Accept uint8 [0,255] or float [0,1], resize to 224×224, apply ImageNet norm."""
        x = x.float()
        if x.max() > 1.5:          # uint8 path
            x = x / 255.0
        x = F.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
        return (x - self.img_mean) / self.img_std

    @torch.no_grad()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return spatial patch features. Input (B, 3, H, W) → output (B, 384, 16, 16)."""
        x = self.preprocess(x)
        out = self.dino(pixel_values=x)     # last_hidden_state: (B, 257, 384)
        patches = out.last_hidden_state[:, 1:]  # drop CLS → (B, 256, 384)
        B = patches.shape[0]
        return patches.reshape(B, self.PATCH_H, self.PATCH_W, 384).permute(0, 3, 1, 2)


# ─── ConvNet spatial adapter ───────────────────────────────────────────────────
class ConvAdapter(nn.Module):
    """Compress DINOv2 patch features: (B, 384, 16, 16) → (B, 128, 4, 4)."""

    def __init__(self, d_model: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(384, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.GELU(),
            nn.Conv2d(256, d_model, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(d_model),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)   # (B, d_model, 4, 4)


# ─── 2D positional embedding ───────────────────────────────────────────────────
class PosEmbed2D(nn.Module):
    """Learnable 2D positional embedding added to flattened spatial tokens."""

    def __init__(self, n_tokens: int, d_model: int):
        super().__init__()
        self.emb = nn.Parameter(torch.zeros(1, n_tokens, d_model))
        nn.init.trunc_normal_(self.emb, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.emb   # (B, n_tokens, d_model)


# ─── World model ───────────────────────────────────────────────────────────────
class PushTWorldModel(nn.Module):
    """
    DINOv2 world model for PushT. Implements the Costable protocol.

    State representation: (B, N_TOKENS, D) spatial token sequence.

    encode_frame(pixels)       → state_tokens (B, 16, D)
    predict(state_tokens, act) → next_state_tokens (B, 16, D)
    get_cost(info, actions)    → costs (B, N) for CEMSolver
    """

    N_TOKENS = 4 * 4       # spatial tokens after ConvAdapter
    D_MODEL  = 128

    def __init__(self):
        super().__init__()
        self.encoder    = DinoEncoder()
        self.conv       = ConvAdapter(self.D_MODEL)
        self.pos_emb    = PosEmbed2D(self.N_TOKENS, self.D_MODEL)
        self.action_proj = nn.Sequential(
            nn.Linear(2, self.D_MODEL),
            nn.GELU(),
            nn.Linear(self.D_MODEL, self.D_MODEL),
        )
        self.transformer = nn.TransformerEncoderLayer(
            d_model=self.D_MODEL,
            nhead=4,
            dim_feedforward=256,
            dropout=0.0,
            activation='gelu',
            batch_first=True,
            norm_first=True,   # pre-norm for stability
        )

    # -- Building blocks -------------------------------------------------------

    def encode_frame(self, pixels: torch.Tensor) -> torch.Tensor:
        """pixels (B, 3, H, W) → state_tokens (B, N_TOKENS, D)."""
        feat = self.encoder(pixels)           # (B, 384, 16, 16)
        feat = self.conv(feat)                # (B, D, 4, 4)
        tokens = feat.flatten(2).permute(0, 2, 1)  # (B, 16, D)
        return self.pos_emb(tokens)           # (B, 16, D)

    def predict(self, state_tokens: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
        """state_tokens (B, 16, D) + action (B, 2) → next_state_tokens (B, 16, D)."""
        act_emb = self.action_proj(action).unsqueeze(1)  # (B, 1, D)
        conditioned = state_tokens + act_emb              # broadcast over tokens
        return self.transformer(conditioned)              # (B, 16, D)

    def pool(self, state_tokens: torch.Tensor) -> torch.Tensor:
        """state_tokens (B, 16, D) → emb (B, D) via mean pooling."""
        return state_tokens.mean(dim=1)

    # -- Training interface ---------------------------------------------------

    def rollout_train(
        self,
        init_tokens: torch.Tensor,
        actions: torch.Tensor,
    ) -> torch.Tensor:
        """
        Multi-step rollout for training.

        init_tokens : (B, 16, D)  — encoded frame at t=0
        actions     : (B, T-1, 2) — actions a_0 .. a_{T-2}

        Returns predicted_tokens (B, T-1, 16, D) = predictions for t=1..T-1.
        """
        T_minus_1 = actions.shape[1]
        state = init_tokens
        preds = []
        for t in range(T_minus_1):
            state = self.predict(state, actions[:, t])  # (B, 16, D)
            preds.append(state)
        return torch.stack(preds, dim=1)   # (B, T-1, 16, D)

    # -- CEMSolver / Costable interface ----------------------------------------

    def get_cost(
        self,
        info_dict: dict,
        action_candidates: torch.Tensor,
    ) -> torch.Tensor:
        """
        Called by CEMSolver at each iteration.

        info_dict['pixels'] : (B, N, 1, 3, H, W)  — current obs, already CHW float
        info_dict['goal']   : (B, N, 1, 3, H, W)  — goal obs
        action_candidates   : (B, N, H, 2)         — action sequences to evaluate

        Returns cost (B, N): lower = better match to goal.
        """
        B, N, horizon = action_candidates.shape[:3]

        # ── Encode initial obs once per planning call (cached) ──────────────
        if 'cached_state_tokens' not in info_dict:
            pixels_0 = info_dict['pixels'][:, 0, 0].to(device)  # (B, 3, H, W)
            with torch.no_grad():
                state_tokens = self.encode_frame(pixels_0)       # (B, 16, D)
            info_dict['cached_state_tokens'] = state_tokens
        state_tokens_0 = info_dict['cached_state_tokens']  # (B, 16, D)

        # ── Encode goal once per planning call (cached) ──────────────────────
        if 'cached_goal_emb' not in info_dict:
            goal_0 = info_dict['goal'][:, 0, 0].to(device)      # (B, 3, H, W)
            with torch.no_grad():
                goal_emb = self.pool(self.encode_frame(goal_0))  # (B, D)
            info_dict['cached_goal_emb'] = goal_emb
        goal_emb = info_dict['cached_goal_emb']  # (B, D)

        # ── Expand initial state for all N samples ───────────────────────────
        # (B, 16, D) → (B*N, 16, D)
        state = state_tokens_0.unsqueeze(1).expand(B, N, -1, -1)
        state = state.reshape(B * N, self.N_TOKENS, self.D_MODEL)

        # ── Multi-step rollout ───────────────────────────────────────────────
        # action_candidates: (B, N, H, 2)
        acts = action_candidates.to(device).reshape(B * N, horizon, 2)
        for h in range(horizon):
            state = self.predict(state, acts[:, h])   # (B*N, 16, D)

        # ── Cost: cosine distance to goal embedding ──────────────────────────
        final_emb = self.pool(state).reshape(B, N, self.D_MODEL)   # (B, N, D)
        goal_exp  = goal_emb.unsqueeze(1).expand_as(final_emb)     # (B, N, D)
        cost = 1.0 - F.cosine_similarity(final_emb, goal_exp, dim=-1)  # (B, N)

        return cost


# ── Quick shape sanity check ──────────────────────────────────────────────────
model = PushTWorldModel().to(device)

_dummy = torch.zeros(2, 3, 64, 64, device=device)
_toks  = model.encode_frame(_dummy)
_act   = torch.zeros(2, 2, device=device)
_next  = model.predict(_toks, _act)
print(f'encode_frame  → {_toks.shape}')   # (2, 16, 128)
print(f'predict       → {_next.shape}')   # (2, 16, 128)
print(f'pool          → {model.pool(_next).shape}')  # (2, 128)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'\nTrainable params: {trainable:,} / {total:,} total')

In [ ]:
# Cell 5: Training
import time

NUM_EPOCHS = 30
LR         = 3e-4
CKPT_PATH  = 'pusht_worldmodel.pt'

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)


def encode_sequence(pixels: torch.Tensor) -> torch.Tensor:
    """Encode all T frames in a batch without gradients (DINOv2 frozen).

    pixels: (B, T, C, H, W) uint8 or float.
    Returns target_tokens: (B, T, N_TOKENS, D).
    """
    B, T, C, H, W = pixels.shape
    pixels_flat = pixels.reshape(B * T, C, H, W).to(device)
    with torch.no_grad():
        tokens_flat = model.encode_frame(pixels_flat)   # (B*T, 16, D)
    return tokens_flat.reshape(B, T, model.N_TOKENS, model.D_MODEL)


model.train()
history = []

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    t0 = time.time()

    for batch in loader:
        pixels  = batch['pixels']    # (B, T, 3, 64, 64)
        actions = batch['action'].to(device)   # (B, T, 2)

        # Encode all frames with frozen DINOv2 (no grad)
        target_tokens = encode_sequence(pixels)          # (B, T, 16, 128)

        # Multi-step rollout: predict t=1..T-1 from t=0 using ground-truth actions
        init_tokens  = target_tokens[:, 0]               # (B, 16, 128)
        action_seq   = actions[:, :-1]                   # (B, T-1, 2)
        pred_tokens  = model.rollout_train(init_tokens, action_seq)  # (B, T-1, 16, 128)

        # MSE in DINOv2 embedding space
        targets = target_tokens[:, 1:].to(device)        # (B, T-1, 16, 128)
        loss = F.mse_loss(pred_tokens, targets.detach())

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()

    scheduler.step()
    avg_loss = epoch_loss / len(loader)
    history.append(avg_loss)
    elapsed = time.time() - t0
    print(f'Epoch {epoch+1:3d}/{NUM_EPOCHS}  loss={avg_loss:.5f}  ({elapsed:.1f}s)')

torch.save(model.state_dict(), CKPT_PATH)
print(f'\nCheckpoint saved to {CKPT_PATH}')

In [ ]:
# Cell 6: Loss curve
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(range(1, NUM_EPOCHS + 1), history, marker='o', markersize=3)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('World Model Training Loss')
plt.tight_layout()
plt.savefig('training_loss.png', dpi=120)
plt.show()

In [ ]:
# Cell 7: Evaluation with MPC (CEMSolver + WorldModelPolicy)

# Load checkpoint (allows re-running eval without retraining)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()

# Image transform: WorldModelPolicy uses this to convert env's HWC uint8
# to CHW float [0,1] before passing to the model.
image_transform = T.ConvertImageDtype(torch.float32)

solver = CEMSolver(
    model=model,
    num_samples=200,
    n_steps=10,
    topk=20,
    var_scale=0.5,
    device=device,
)
config = PlanConfig(
    horizon=10,
    receding_horizon=2,
    history_len=1,
    action_block=1,
    warm_start=True,
)
policy = WorldModelPolicy(
    solver=solver,
    config=config,
    transform={'pixels': image_transform, 'goal': image_transform},
)

NUM_ENVS = 4
eval_world = swm.World('swm/PushT-v1', num_envs=NUM_ENVS, image_shape=(64, 64))
eval_world.set_policy(policy)

print('Evaluating...')
results = eval_world.evaluate(episodes=20, seed=42)
eval_world.close()

print(f"\nSuccess rate:    {results['success_rate']:.1f}%")
print(f"Per-episode:     {results['episode_successes']}")

In [ ]:
# Cell 8: Random-policy baseline for comparison
baseline_world = swm.World('swm/PushT-v1', num_envs=NUM_ENVS, image_shape=(64, 64))
baseline_world.set_policy(swm.policy.RandomPolicy(seed=0))
baseline = baseline_world.evaluate(episodes=20, seed=42)
baseline_world.close()

print(f"Random baseline: {baseline['success_rate']:.1f}%")
print(f"Our WM policy:   {results['success_rate']:.1f}%")
delta = results['success_rate'] - baseline['success_rate']
print(f"Improvement:     {delta:+.1f}pp")

In [ ]:
# Cell 9: Visualise a planned trajectory
import numpy as np

# Re-collect one episode with the WM policy for rendering
viz_world = swm.World('swm/PushT-v1', num_envs=1, image_shape=(64, 64))
viz_world.set_policy(
    WorldModelPolicy(
        solver=CEMSolver(model=model, num_samples=200, n_steps=10, topk=20,
                         var_scale=0.5, device=device),
        config=config,
        transform={'pixels': image_transform, 'goal': image_transform},
    )
)
viz_world.collect('data/viz_episode.lance', episodes=1, seed=7, mode='overwrite')
viz_world.close()

# Load and display
viz_ds = swm.data.load_dataset(
    os.path.abspath('data/viz_episode.lance'),
    num_steps=64,
    keys_to_load=['pixels'],
)
pixels = viz_ds[0]['pixels']   # (T, 3, 64, 64)

n_show = min(32, pixels.shape[0])
n_cols = 8
n_rows = math.ceil(n_show / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.5, n_rows * 1.5))

for i, ax in enumerate(axes.flatten()):
    if i < n_show:
        img = pixels[i * (pixels.shape[0] // n_show)].permute(1, 2, 0).numpy()
        if img.max() <= 1.0:
            img = (img * 255).astype(np.uint8)
        else:
            img = img.astype(np.uint8)
        ax.imshow(img)
        ax.set_title(f't={i * (pixels.shape[0] // n_show)}', fontsize=7)
    ax.axis('off')

plt.suptitle('WM-policy trajectory', fontsize=10)
plt.tight_layout()
plt.savefig('wm_trajectory.png', dpi=120)
plt.show()
print('Saved to wm_trajectory.png')